## 1. Импорты, подключение, параметры

In [ ]:
import pickle

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score, roc_auc_score
from sqlalchemy import text
from sqlalchemy.dialects.postgresql import ARRAY, BIGINT

from database import postgres_connection

engine = postgres_connection()

N_CLUSTERS = 16     # ровно столько dist_cluster_i ждёт app.py (n_clusters = 16 в load_features)
N_SVD = 64           # промежуточная размерность текста перед кластеризацией (сам вектор в признаки не идёт)
N_TRAIN_ROWS = 3_500_000
AGE_BINS = [0, 17, 25, 35, 150]
AGE_LABELS = ['teens', 'youth', 'adults', 'seniors']

## 2. Загрузка данных и разделение по времени

In [ ]:
df_user_data = pd.read_sql("SELECT * FROM public.user_data", con=engine)
df_post_text = pd.read_sql("SELECT * FROM public.post_text_df", con=engine)
if 'post_id' not in df_post_text.columns and 'id' in df_post_text.columns:
    df_post_text = df_post_text.rename(columns={'id': 'post_id'})

df_feed_data = pd.read_sql(
    "SELECT timestamp, user_id, post_id, action, target FROM public.feed_data "
    "WHERE action = 'view' AND target IS NOT NULL ORDER BY timestamp "
    f"LIMIT {N_TRAIN_ROWS}",
    con=engine,
)
df_feed_data['timestamp'] = pd.to_datetime(df_feed_data['timestamp'])
df_feed_data['target'] = df_feed_data['target'].astype('int8')

print(df_user_data.shape, df_post_text.shape, df_feed_data.shape)

In [ ]:
df_feed_data = df_feed_data.sort_values('timestamp', kind='stable').reset_index(drop=True)
split_idx = int(len(df_feed_data) * 0.8)
feed_train = df_feed_data.iloc[:split_idx].reset_index(drop=True)
feed_test = df_feed_data.iloc[split_idx:].reset_index(drop=True)
print('train:', feed_train.shape, 'test:', feed_test.shape)

## 3. Признаки пользователей

`age_group` — тот же бакетинг, что уже использовался в проекте.
`country_city_count` — сколько пользователей живут в той же паре (country, city): простой признак
«насколько типичное» сочетание страны и города у человека.

In [ ]:
df_user_data['age_group'] = pd.cut(df_user_data['age'], bins=AGE_BINS, labels=AGE_LABELS).astype(str)

city_counts = df_user_data.groupby(['country', 'city'])['user_id'].transform('count')
df_user_data['country_city_count'] = city_counts.astype('uint16')

USER_FEATURE_COLS = ['user_id', 'gender', 'age', 'country', 'city', 'exp_group', 'os', 'source',
                      'age_group', 'country_city_count']
user_static = df_user_data[USER_FEATURE_COLS].copy()
user_static.head()

## 4. Признаки постов: текст, кластеризация

Текст векторизуется через TF-IDF и понижается до `N_SVD` компонент — это только промежуточный шаг для
кластеризации, сам вектор в финальные признаки не идёт (`app.py` его не ждёт). `KMeans(n_clusters=16)`
даёт метку кластера (`text_cluster`) и ровно 16 расстояний до центров (`dist_cluster_0..15`) —
это и есть компактная замена эмбеддингу поста, которую понимает текущий сервис.

In [ ]:
df_post_text['text'] = df_post_text['text'].fillna('').astype(str)
df_post_text['topic'] = df_post_text['topic'].fillna('unknown').astype(str)
df_post_text['text_len'] = df_post_text['text'].str.len().astype('int32')
df_post_text['word_count'] = df_post_text['text'].str.split().str.len().fillna(0).astype('int32')
df_post_text['avg_word_len'] = (
    (df_post_text['text_len'] / df_post_text['word_count'].replace(0, np.nan))
    .fillna(0.0).astype('float32')
)

In [ ]:
tfidf = TfidfVectorizer(max_features=30000, min_df=2, max_df=0.95, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_post_text['text'])

svd_model = TruncatedSVD(n_components=N_SVD, random_state=42)
post_vectors = svd_model.fit_transform(tfidf_matrix).astype('float32')

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
text_cluster = kmeans.fit_predict(post_vectors).astype('uint8')
dist_cluster = kmeans.transform(post_vectors).astype('float32')   # (n_posts, N_CLUSTERS) — ровно dist_cluster_0..15

print('объяснённая дисперсия SVD:', round(svd_model.explained_variance_ratio_.sum(), 3))
print('размеры кластеров:', np.bincount(text_cluster))

In [ ]:
dist_cols = [f'dist_cluster_{i}' for i in range(N_CLUSTERS)]
post_static = df_post_text[['post_id', 'topic', 'text', 'text_len', 'word_count', 'avg_word_len']].copy()
post_static['text_cluster'] = text_cluster
post_static = pd.concat([post_static.reset_index(drop=True),
                          pd.DataFrame(dist_cluster, columns=dist_cols)], axis=1)
post_static.head()

## 5. Временные признаки события

Копия `extract_dt_features` из `app.py`, но векторизованная по всей колонке `timestamp` сразу
(построчно та же формула была бы слишком медленной на миллионах строк). Важно совпадение не только
границ `part_of_day` и сдвига `dayofweek + 1`, но и **типа значений**: `app.py` кладёt в `month`,
`day`, `dayofweek`, `hour` целые числа (обёрнутые в `category`), а в `is_weekend` — `bool`, БЕЗ
приведения к строке. Если здесь обучить модель на строках `"5"`/`"True"`, а сервис будет присылать
число `5`/`True`, CatBoost увидит это как разные категории — признак тихо перестанет работать.
Поэтому ниже типы в точности повторяют `extract_dt_features`, а не унифицируются под строку.

In [ ]:
def build_dt_features(ts: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame(index=ts.index)
    out['year'] = ts.dt.year.astype('uint16')
    out['month'] = ts.dt.month.to_numpy()                      # int, как в app.py (category поверх int)
    out['day'] = ts.dt.day.to_numpy()
    out['dayofweek'] = (ts.dt.dayofweek + 1).to_numpy()
    out['is_weekend'] = ts.dt.dayofweek.isin([5, 6]).to_numpy()  # bool, как в app.py
    out['hour'] = ts.dt.hour.to_numpy()

    part_of_day = pd.Series('night', index=ts.index)
    part_of_day[(ts.dt.hour >= 6) & (ts.dt.hour < 12)] = 'morning'
    part_of_day[(ts.dt.hour >= 12) & (ts.dt.hour < 18)] = 'afternoon'
    part_of_day[(ts.dt.hour >= 18) & (ts.dt.hour < 24)] = 'evening'
    out['part_of_day'] = part_of_day.to_numpy()                 # строка — здесь app.py тоже строку кладёт
    return out

## 6. Сборка обучающей выборки

Повторяем ТОТ ЖЕ порядок, которым `app.py` собирает `df_final`: сначала временные признаки, затем
признаки пользователя, затем признаки поста, в конце убираем `user_id`, `post_id`, `text`. Итоговый
порядок колонок у `X_train` должен совпадать с тем, что сервис строит на лету.

In [ ]:
DT_COLS = ['year', 'month', 'day', 'dayofweek', 'is_weekend', 'hour', 'part_of_day']
# категориальные с ТЕКСТОВЫМИ значениями — тут безопасно унифицировать под строку
TEXT_CAT_COLS = ['country', 'city', 'os', 'source', 'age_group', 'topic', 'part_of_day']
# категориальные из extract_dt_features, где app.py хранит int/bool как есть — значения НЕ трогаем
DT_CAT_COLS = ['month', 'day', 'dayofweek', 'is_weekend', 'hour']
CAT_COLS = DT_CAT_COLS + TEXT_CAT_COLS


def assemble(feed: pd.DataFrame) -> pd.DataFrame:
    dt_feats = build_dt_features(feed['timestamp'])
    df = pd.concat([dt_feats, feed[['user_id', 'post_id', 'target']].reset_index(drop=True)], axis=1)
    df = df.merge(user_static, on='user_id', how='inner')
    df = df.merge(post_static.drop(columns=['text']), on='post_id', how='inner')
    return df


train_set = assemble(feed_train)
test_set = assemble(feed_test)

FEATURE_COLS = DT_COLS + [c for c in USER_FEATURE_COLS if c != 'user_id'] + \
    [c for c in post_static.columns if c not in ('post_id', 'text')]

for df in (train_set, test_set):
    for col in TEXT_CAT_COLS:
        df[col] = df[col].astype(str)
    # DT_CAT_COLS намеренно НЕ приводим к строке: int/bool должны совпадать по типу с тем,
    # что на лету строит extract_dt_features в app.py (см. ячейку выше)
    for col in FEATURE_COLS:
        if col not in CAT_COLS:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('float32')

print('признаков:', len(FEATURE_COLS))
print(FEATURE_COLS)
print('train/test:', train_set.shape, test_set.shape)

## 7. Обучение CatBoost

In [ ]:
cut = int(len(train_set) * 0.9)
train_pool = Pool(train_set.loc[:cut - 1, FEATURE_COLS], train_set.loc[:cut - 1, 'target'], cat_features=CAT_COLS)
val_pool = Pool(train_set.loc[cut:, FEATURE_COLS], train_set.loc[cut:, 'target'], cat_features=CAT_COLS)

model = CatBoostClassifier(
    iterations=2000, depth=6, learning_rate=0.06, l2_leaf_reg=3,
    eval_metric='AUC', random_seed=42, verbose=100,
)
model.fit(train_pool, eval_set=val_pool, use_best_model=True, early_stopping_rounds=100)
print('деревьев в модели:', model.tree_count_)

## 8. Оценка качества

ROC-AUC/PR-AUC на отложенных просмотрах, затем HitRate@5 «как у сервиса»: для каждого пользователя
ранжируются все посты, которые он ещё не лайкал на момент среза, и проверяется, попал ли хоть один
реально лайкнутый им (в тестовом периоде) пост в топ-5.

In [ ]:
proba = model.predict_proba(test_set[FEATURE_COLS])[:, 1]
print('ROC-AUC:', round(roc_auc_score(test_set['target'], proba), 4))
print('PR-AUC: ', round(average_precision_score(test_set['target'], proba), 4))

In [ ]:
def hitrate_like_service(feed_train, feed_test, k=5, n_users=1500, seed=42):
    rng = np.random.default_rng(seed)
    liked_before = feed_train.loc[feed_train['target'] == 1].groupby('user_id')['post_id'].agg(set)
    liked_in_test = feed_test.loc[feed_test['target'] == 1].groupby('user_id')['post_id'].agg(set)

    candidate_users = liked_in_test.index.intersection(user_static['user_id'])
    users = rng.choice(candidate_users.to_numpy(), size=min(n_users, len(candidate_users)), replace=False)

    dt_sample = feed_test['timestamp'].sample(1, random_state=seed).iloc[0]
    dt_feats = build_dt_features(pd.Series([dt_sample]))   # одна строка, как в app.py (один dt на запрос)

    hits = 0
    for uid in users:
        u_row = user_static.loc[user_static['user_id'] == uid]
        liked_ids = liked_before.get(uid, set())
        candidates = post_static[~post_static['post_id'].isin(liked_ids)]
        if candidates.empty:
            candidates = post_static

        frame = pd.merge(
            pd.merge(dt_feats, u_row, how='cross'),
            candidates, how='cross',
        ).drop(columns=['user_id', 'post_id', 'text'])
        for col in TEXT_CAT_COLS:
            frame[col] = frame[col].astype(str)
        scores = model.predict_proba(frame[FEATURE_COLS])[:, 1]

        top_ids = candidates['post_id'].to_numpy()[np.argsort(-scores)[:k]]
        hits += len(set(top_ids) & liked_in_test[uid]) > 0

    return hits / len(users)


hr5 = hitrate_like_service(feed_train, feed_test, k=5)
print(f'HitRate@5 как у сервиса: {hr5:.4f}')

## 9. Таблицы для сервиса

Схема и типы колонок повторяют то, что `load_features()` в `app.py` ожидает прочитать из БД
(`rec_user_features`, `rec_post_features`, `rec_user_likes`). `liked_posts` сохраняется как
Postgres `ARRAY(BIGINT)`, чтобы при чтении psycopg2 сразу отдавал Python-список, а не строку.

In [ ]:
final_user_features = user_static.copy()
final_user_features['user_id'] = final_user_features['user_id'].astype('uint32')
final_user_features['gender'] = final_user_features['gender'].astype('uint8')
final_user_features['age'] = final_user_features['age'].astype('uint8')
final_user_features['exp_group'] = final_user_features['exp_group'].astype('uint8')
for c in ('country', 'city', 'os', 'source', 'age_group'):
    final_user_features[c] = final_user_features[c].astype(str)
final_user_features['country_city_count'] = final_user_features['country_city_count'].astype('uint16')

In [ ]:
final_post_features = post_static.copy()
final_post_features['post_id'] = final_post_features['post_id'].astype('uint16')
final_post_features['topic'] = final_post_features['topic'].astype(str)
final_post_features['text'] = final_post_features['text'].astype(str)
final_post_features['text_cluster'] = final_post_features['text_cluster'].astype('uint8')
final_post_features['text_len'] = final_post_features['text_len'].astype('uint32')
final_post_features['word_count'] = final_post_features['word_count'].astype('uint16')
final_post_features['avg_word_len'] = final_post_features['avg_word_len'].astype('float16')
for c in dist_cols:
    final_post_features[c] = final_post_features[c].astype('float16')

In [ ]:
# Лайки пользователя — по ВСЕЙ истории (train + test), сервису нужно знать все лайки на сегодня,
# а не только те, что попали в обучающую выборку.
all_likes = df_feed_data.loc[df_feed_data['target'] == 1].groupby('user_id')['post_id'].agg(list)
final_user_likes = pd.DataFrame({
    'user_id': user_static['user_id'],
    'liked_posts': user_static['user_id'].map(all_likes).apply(lambda v: v if isinstance(v, list) else []),
})
final_user_likes['user_id'] = final_user_likes['user_id'].astype('uint32')
final_user_likes.head()

In [ ]:
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)


def save_table(df, name, array_cols=None):
    dtype = {col: ARRAY(BIGINT) for col in (array_cols or [])}
    df.to_sql(name, con=engine, if_exists='replace', index=False,
              method='multi', chunksize=max(1, 30000 // df.shape[1]), dtype=dtype)


save_table(final_user_features, 'rec_user_features')
save_table(final_post_features, 'rec_post_features')
save_table(final_user_likes, 'rec_user_likes', array_cols=['liked_posts'])
print('Сохранены model.pkl, rec_user_features, rec_post_features, rec_user_likes.')

## 10. Проверка запущенного сервиса

In [ ]:
import requests

test_user = int(feed_test['user_id'].iloc[0])
test_dt = feed_test['timestamp'].iloc[0].isoformat()

response = requests.get(
    "http://127.0.0.1:8000/post/recommendations/",
    params={"user_id": test_user, "dt": test_dt, "limit": 5},
)
print(f"Рекомендации для user_id = {test_user}:")
for rec in response.json():
    print(rec["id"], rec["topic"], rec["text"][:80])